# Final Evaluation Summary

This notebook summarizes the saved evaluation results for:

- Model A — Intent Classifier
- Model B — Extractive QA
- Model C — Technical Support Specialist
- Router Comparison
- End-to-End Regression Testing

The notebook reads the final JSON reports and does not retrain the models.

In [1]:
import json
from pathlib import Path

import pandas as pd


REPORTS = Path("../reports")


def load_json(filename):

    with open(
        REPORTS / filename,
        "r",
        encoding="utf-8",
    ) as file:

        return json.load(file)


model_a = load_json(
    "model_a_results.json"
)

model_b = load_json(
    "model_b_results.json"
)

model_c = load_json(
    "model_c_results.json"
)

router = load_json(
    "router_results.json"
)

e2e = load_json(
    "e2e_regression_results.json"
)

## Model Comparison

In [2]:
model_summary = pd.DataFrame(
    [
        {
            "Model":
                "Model A - Intent Classifier",

            "Baseline":
                (
                    f"Accuracy = "
                    f"{model_a['baseline']['accuracy']:.3f}, "
                    f"Macro F1 = "
                    f"{model_a['baseline']['f1_macro']:.3f}"
                ),

            "Final":
                (
                    f"Accuracy = "
                    f"{model_a['fine_tuned_test']['accuracy']:.3f}, "
                    f"Macro F1 = "
                    f"{model_a['fine_tuned_test']['f1_macro']:.3f}"
                ),
        },

        {
            "Model":
                "Model B - Extractive QA",

            "Baseline":
                (
                    f"EM = "
                    f"{model_b['baseline']['exact_match']:.3f}, "
                    f"F1 = "
                    f"{model_b['baseline']['token_f1']:.3f}"
                ),

            "Final":
                (
                    f"EM = "
                    f"{model_b['experiment_3']['exact_match']:.3f}, "
                    f"F1 = "
                    f"{model_b['experiment_3']['token_f1']:.3f}"
                ),
        },

        {
            "Model":
                "Model C - Support Specialist",

            "Baseline":
                (
                    f"PPL = "
                    f"{model_c['baseline']['perplexity']:.3f}, "
                    f"ROUGE-L = "
                    f"{model_c['baseline']['rouge_l']:.3f}"
                ),

            "Final":
                (
                    f"PPL = "
                    f"{model_c['exp6_final']['perplexity']:.3f}, "
                    f"ROUGE-L = "
                    f"{model_c['exp6_final']['rouge_l']:.3f}, "
                    f"Golden Set = "
                    f"{model_c['exp6_final']['golden_set_pass_rate']:.2%}"
                ),
        },
    ]
)

model_summary

,Model,Baseline,Final
0,Model A - Intent Classifier,"Accuracy = 0.125, Macro F1 = 0.028","Accuracy = 0.950, Macro F1 = 0.949"
1,Model B - Extractive QA,"EM = 0.000, F1 = 0.416","EM = 1.000, F1 = 1.000"
2,Model C - Support Specialist,"PPL = 25.209, ROUGE-L = 0.136","PPL = 2.940, ROUGE-L = 0.442, Golden Set = 83.33%"


## Router Comparison

In [3]:
router_summary = pd.DataFrame(
    [
        {
            "Router":
                "Rules + Classifier",

            **router[
                "rules_classifier"
            ],
        },

        {
            "Router":
                "LLM Router",

            **router[
                "llm_router"
            ],
        },

        {
            "Router":
                "Hybrid Router",

            **router[
                "hybrid_router"
            ],
        },
    ]
)

router_summary

,Router,accuracy,macro_f1,avg_latency_ms,json_valid_rate,fallback_rate,confidence_threshold
0,Rules + Classifier,0.916667,0.914286,27.984232,NaN,NaN,NaN
1,LLM Router,0.250000,0.115385,802.324021,1.0,NaN,NaN
2,Hybrid Router,1.000000,1.000000,137.488559,NaN,0.166667,0.25


## Selected Production Router

The Hybrid Router was selected for production.

The final policy is:

1. Hard rules are evaluated first.
2. Model A predicts the intent.
3. If classifier confidence is greater than or equal to `0.25`, the classifier route is used.
4. Otherwise, Model C is used as the LLM router fallback.

The router scores reported here are based only on the current 12-case evaluation set.

## End-to-End Regression

In [4]:
print(
    f"Passed: "
    f"{e2e['passed']}"
    f"/"
    f"{e2e['total_tests']}"
)

print(
    f"Failed: "
    f"{e2e['failed']}"
)


e2e_results_df = pd.DataFrame(
    e2e["tests"]
)

e2e_results_df[
    [
        "test",
        "status",
    ]
]

Passed: 13/13
Failed: 0


,test,status
0,Documentation QA route,PASS
1,Documentation grounded answer,PASS
2,Database issue routes to tools,PASS
3,Database diagnostics used,PASS
4,Database grounded response,PASS
5,QLoRA support route,PASS
6,QLoRA uses LLM fallback,PASS
7,QLoRA regression grounding,PASS
8,Corruption routes to escalation,PASS
9,Human escalation created,PASS


## Final Findings

### Model A

Fine-tuning improved the intent classifier from a random-like baseline to a strong classifier result:

- Baseline Accuracy: 12.5%
- Final Accuracy: 95.0%
- Final Macro F1: 0.9495

### Model B

Fine-tuning significantly improved extractive QA:

- Baseline EM: 0.00
- Baseline Token F1: 0.4162
- Final EM: 1.00
- Final Token F1: 1.00

The final test set contains only 8 QA pairs, so the perfect score should be interpreted carefully.

### Model C

Model C improved across general generation and behavioral evaluation:

- Baseline Perplexity: 25.2089
- Final Perplexity: 2.9399
- Baseline ROUGE-L: 0.1363
- Final ROUGE-L: 0.4422
- Final Golden Set: 83.33%

The remaining model-only weakness was explicit uncertainty handling, which is handled by a deterministic agent guardrail.

### Router

The Hybrid Router achieved the strongest result on the current 12-case router test set while using the LLM only for low-confidence cases.

### End-to-End System

The final regression suite passed:

**13 / 13 tests**